# 1. Решение

In [2]:
from rdkit import Chem
from meeko import MoleculePreparation
from meeko import PDBQTWriterLegacy
from pathlib import Path

In [3]:
def create_pdbqt(mol, name_mol):

    '''Запись .pdbqt'''
    
    mk_prep = MoleculePreparation(min_ring_size=6, rigid_macrocycles=False) # Обрабатываются алифатические циклы длины >=6 по дефолту
    molsetup_list = mk_prep(mol)

    molsetup = molsetup_list[0]

    pdbqt_string = PDBQTWriterLegacy.write_string(molsetup)

    with open(f'results/{name_mol}.pdbqt', 'w') as f: 
        f.write(pdbqt_string[0])    


def smi_prep(path_smi):

    '''Подготовка входных данных из .smi файла'''
    
    with open(path_smi, 'r') as f:
        smi_mols = {}
        f.readline()
        for line in f:
            smi, key = line.strip().split()
            smi_mols[key] = smi
    
    for name_mol, smi_mol in smi_mols.items():
        
        mol = Chem.MolFromSmiles(smi_mol)
        mol.SetProp('_Name', name_mol)
        mol = Chem.AddHs(mol)
        Chem.AllChem.EmbedMolecule(mol, randomSeed = 42)
        create_pdbqt(mol, name_mol)
        
def sdf_prep(path_sdf):

    '''Подготовка входных данных из .sdf файла'''
    
    molecules = Chem.SDMolSupplier(path_sdf)
    for mol in molecules:
        
        mol = Chem.AddHs(mol)
        name_mol = mol.GetProp('_Name')
        Chem.AllChem.EmbedMolecule(mol, randomSeed = 42)
        create_pdbqt(mol, name_mol)

def read_and_create(path):

    '''Управляющая функция,
    которая запускает обработку файла в заивисимости от формата.'''
    
    file_path = Path(path)
    ext = file_path.suffix
    
    if not Path("results").exists(): # Папка с аутпутом
        Path("results").mkdir()
    
    if ext == '.smi':
        smi_prep(path)
    elif ext == '.sdf':
        sdf_prep(path)
    else:
        raise NameError('Wrong extension. Should be SMI or SDF')

read_and_create('vina_data/examples.sdf')